# Stage 1 — Synthetic Patient Generation with Gold Criterion Labels

Inverts the annotation problem: instead of labeling real (patient, criterion) pairs
(silver, unverifiable), we **sample a target label spec first and generate the patient
note to realize it**. Labels are gold by construction — which makes criterion assessment
a verifiable-reward task for GRPO in Stage 2.

Claude is used here for *data generation only* (allowed); no closed model touches
the production/eval inference path.

**Pipeline:**
1. Sample specs: per trial, pick criteria and assign target labels with a rebalanced
   class mix (30% `excluded` on exclusion criteria vs 2% in natural data)
2. Generate: Claude writes a realistic patient note realizing the spec (JSON out,
   `feasible: false` escape hatch for contradictory specs)
3. Verify: a *different* model (Haiku) blind-labels each (note, criterion) pair;
   pairs where blind label == target label get `verified: true`

**Outputs (Drive):**
- `synthetic_patients.jsonl` — one record per generated patient (note + spec)
- `synthetic_criteria_gold.jsonl` — one record per (patient, criterion) pair,
  join to patients on `syn_id`

**Contamination:** trials judged in TREC22 are excluded from generation by default.

**Cost estimate (defaults):** ~6k Sonnet generations (~1k in / 700 out each) +
~40k Haiku verifications (~700 in / 10 out each) ≈ $30–50 total.

**Both output files are resumable** — generation and verification checkpoint by id.

In [ ]:
!pip install -q anthropic tqdm

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = ''  # never commit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT     = '/content/drive/MyDrive/ct_data23'
CRITERIA_PATH = f'{DATA_ROOT}/criteria_data.jsonl'
QRELS_PATH    = f'{DATA_ROOT}/unified_qrels.jsonl'
PATIENTS_PATH = f'{DATA_ROOT}/synthetic_patients.jsonl'
PAIRS_PATH    = f'{DATA_ROOT}/synthetic_criteria_gold.jsonl'

GEN_MODEL     = 'claude-sonnet-4-6'   # writes patient notes
VERIFY_MODEL  = 'claude-haiku-4-5'    # blind re-labeling (different model = independent check)

N_PATIENTS_PER_TRIAL  = 4
MAX_INC_PER_SPEC      = 5
MAX_EXC_PER_SPEC      = 3
MIN_CRIT_CHARS        = 15    # drop parser junk fragments
MAX_CRIT_CHARS        = 300
CONCURRENCY           = 12
SEED                  = 42
EXCLUDE_TREC22_TRIALS = True

# Target label mix per criterion — deliberately oversamples the rare classes
# (natural R1 data: excluded 1.9%, not_included 12.4%)
INC_LABEL_P = {'included': 0.45, 'not_included': 0.30, 'not_enough_information': 0.25}
EXC_LABEL_P = {'not_excluded': 0.45, 'excluded': 0.30, 'not_enough_information': 0.25}

# Note style variety — assessor must not overfit to one register
STYLES = [
    ('trec_narrative', 0.4),   # case-vignette prose like TREC topics
    ('ehr_structured', 0.4),   # HPI / PMH / Meds / Labs sections
    ('referral_brief', 0.2),   # terse 3-5 sentence referral
]

In [ ]:
import json

# Trials judged in TREC22 → excluded from generation (contamination guard)
trec22_trials = set()
example_trec_topic = None
with open(QRELS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if rec['source'] == 'trec22':
            trec22_trials.add(rec['doc_id'])
        elif rec['source'] == 'trec21' and example_trec_topic is None:
            example_trec_topic = rec['topic_text']  # style few-shot (train split only)

def usable(criteria):
    return [c.strip() for c in criteria
            if MIN_CRIT_CHARS <= len(c.strip()) <= MAX_CRIT_CHARS]

trial_pool = []
n_excluded_contam, n_too_few = 0, 0
with open(CRITERIA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if EXCLUDE_TREC22_TRIALS and rec['nct_id'] in trec22_trials:
            n_excluded_contam += 1
            continue
        inc, exc = usable(rec['include_criteria']), usable(rec['exclude_criteria'])
        if len(inc) + len(exc) < 2:
            n_too_few += 1
            continue
        trial_pool.append({'nct_id': rec['nct_id'], 'inc': inc, 'exc': exc})

print(f'Trial pool           : {len(trial_pool)}')
print(f'Excluded (trec22)    : {n_excluded_contam}')
print(f'Excluded (<2 usable) : {n_too_few}')
print(f'Planned generations  : {len(trial_pool) * N_PATIENTS_PER_TRIAL:,}')

## Spec sampling

Deterministic under `SEED` — specs are rebuilt identically on resume, so the
checkpoint (skip done `syn_id`s) is safe across restarts.

In [ ]:
import random
from collections import Counter

rng = random.Random(SEED)

def weighted_choice(dist, r):
    return r.choices(list(dist.keys()), weights=list(dist.values()), k=1)[0]

def sample_spec(trial, i, r):
    n_inc = r.randint(1, min(MAX_INC_PER_SPEC, len(trial['inc']))) if trial['inc'] else 0
    n_exc = r.randint(0, min(MAX_EXC_PER_SPEC, len(trial['exc']))) if trial['exc'] else 0
    if n_inc + n_exc == 0:
        n_inc = 1
    targets = (
        [{'crit_type': 'inclusion', 'criterion': c,
          'label': weighted_choice(INC_LABEL_P, r)}
         for c in r.sample(trial['inc'], n_inc)] +
        [{'crit_type': 'exclusion', 'criterion': c,
          'label': weighted_choice(EXC_LABEL_P, r)}
         for c in r.sample(trial['exc'], n_exc)]
    )
    return {
        'syn_id':  f"{trial['nct_id']}__{i}",
        'nct_id':  trial['nct_id'],
        'style':   r.choices([s for s, _ in STYLES], weights=[w for _, w in STYLES], k=1)[0],
        'age':     r.randint(19, 84),
        'sex':     r.choice(['male', 'female']),
        'targets': targets,
    }

all_specs = [sample_spec(t, i, rng)
             for t in trial_pool
             for i in range(N_PATIENTS_PER_TRIAL)]

target_dist = Counter(t['label'] for s in all_specs for t in s['targets'])
n_targets   = sum(target_dist.values())
print(f'Specs: {len(all_specs):,}  |  target pairs: {n_targets:,}')
for label, count in sorted(target_dist.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:6,}  ({100*count/n_targets:.1f}%)')

print('\n--- sample spec ---')
s = all_specs[0]
print(f"{s['syn_id']}  style={s['style']}  age={s['age']}  sex={s['sex']}")
for t in s['targets']:
    print(f"  [{t['crit_type'][:3]}] {t['label']:24s} {t['criterion'][:80]}")

## Prompts

The generator must satisfy every target label simultaneously. Two failure modes are
designed around:
- **contradictory specs** (random labels can be clinically impossible together) —
  the model returns `feasible: false` and we drop the spec
- **NEI leakage** (note accidentally determines a criterion targeted as NEI) —
  explicit silence instruction + caught downstream by blind verification

In [ ]:
import re

SYSTEM_GEN = """You are generating synthetic patient notes for clinical trial matching research.

You will be given eligibility criteria from a real clinical trial, each with a TARGET assessment. Write a realistic patient note such that a clinical expert reading it would reach exactly the target assessment for every listed criterion.

Target semantics:
- included: the note contains decisive evidence the patient MEETS this inclusion criterion
- not_included: the note contains decisive evidence the patient does NOT meet this inclusion criterion
- excluded: the note contains decisive evidence this exclusion criterion APPLIES to the patient
- not_excluded: the note contains decisive evidence this exclusion criterion does NOT apply
- not_enough_information: the note must say NOTHING that determines this criterion either way

Rules:
1. Evidence may be implicit but must be decisive — e.g. \"walks two miles daily\" suffices for a performance-status criterion. A careful clinician must land on the target label, not merely find it plausible.
2. For not_enough_information targets, stay completely silent on the relevant attribute. Do not hint.
3. Add realistic unrelated detail (demographics, meds, social history, an incidental comorbidity) so the note does not read as constructed around the criteria.
4. Keep the note internally consistent and clinically plausible for the given age and sex.
5. If the target set is clinically contradictory or cannot be realized in one coherent patient, respond with {\"feasible\": false} and nothing else.

Respond with JSON only, no markdown fences:
{\"feasible\": true, \"patient_note\": \"...\", \"evidence\": [{\"idx\": 0, \"evidence\": \"short quote from the note, or 'omitted' for not_enough_information\"}]}"""

STYLE_INSTR = {
    'trec_narrative': ('Write as a narrative case vignette, one or two paragraphs of flowing '
                       'clinical prose, in the style of this example:\n---\n{example}\n---'),
    'ehr_structured': ('Write as a structured EHR note with sections: HPI, PMH, Medications, '
                       'Social History, and Labs/Vitals where relevant. Telegraphic clinical style.'),
    'referral_brief': ('Write as a brief referral note, 3-5 sentences, terse.'),
}

def make_gen_prompt(spec):
    style = STYLE_INSTR[spec['style']]
    if spec['style'] == 'trec_narrative':
        style = style.format(example=example_trec_topic[:600])
    lines = [f"Patient: {spec['age']}-year-old {spec['sex']}", '', style, '',
             'Criteria and target assessments:']
    for j, t in enumerate(spec['targets']):
        lines.append(f"{j}. [{t['crit_type']}] {t['criterion']}")
        lines.append(f"   TARGET: {t['label']}")
    return '\n'.join(lines)

def extract_json(text):
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

print(make_gen_prompt(all_specs[0]))

In [ ]:
# Sanity check: generate 3 patients, eyeball notes against their specs before the full run
import asyncio
from anthropic import AsyncAnthropic

aclient = AsyncAnthropic()

async def _sanity():
    tasks = [aclient.messages.create(
                 model=GEN_MODEL, max_tokens=1500, system=SYSTEM_GEN,
                 messages=[{'role': 'user', 'content': make_gen_prompt(s)}])
             for s in all_specs[:3]]
    return await asyncio.gather(*tasks)

for spec, resp in zip(all_specs[:3], await _sanity()):
    data = extract_json(resp.content[0].text)
    print(f"\n{'='*70}\n{spec['syn_id']}  style={spec['style']}")
    if data is None or not data.get('feasible'):
        print('  INFEASIBLE or unparseable')
        continue
    print(f"\n{data['patient_note']}\n")
    for t, ev in zip(spec['targets'], data.get('evidence', [])):
        print(f"  {t['label']:24s} {t['criterion'][:60]}")
        print(f"  {'':24s} → {ev.get('evidence', '?')[:80]}")

## Full generation run
Appends to `synthetic_patients.jsonl` in chunks; resumes by `syn_id`.
Unparseable/errored specs are skipped (not written) so a re-run retries them.

In [ ]:
from tqdm.auto import tqdm

done_ids = set()
if os.path.exists(PATIENTS_PATH):
    with open(PATIENTS_PATH) as f:
        for line in f:
            done_ids.add(json.loads(line)['syn_id'])

todo = [s for s in all_specs if s['syn_id'] not in done_ids]
print(f'Done: {len(done_ids):,}  |  Remaining: {len(todo):,}')

sem = asyncio.Semaphore(CONCURRENCY)

async def gen_patient(spec):
    async with sem:
        for attempt in range(3):
            try:
                resp = await aclient.messages.create(
                    model=GEN_MODEL, max_tokens=1500, system=SYSTEM_GEN,
                    messages=[{'role': 'user', 'content': make_gen_prompt(spec)}],
                )
                data = extract_json(resp.content[0].text)
                if data is None:
                    raise ValueError('unparseable JSON')
                return spec, data
            except Exception as e:
                if attempt == 2:
                    return spec, {'feasible': None, 'error': str(e)}
                await asyncio.sleep(2 ** attempt * 2)

CHUNK = 100
n_ok, n_infeasible, n_failed = 0, 0, 0

for start in tqdm(range(0, len(todo), CHUNK), desc='Generating'):
    chunk   = todo[start:start+CHUNK]
    results = await asyncio.gather(*[gen_patient(s) for s in chunk])
    with open(PATIENTS_PATH, 'a') as out_f:
        for spec, data in results:
            if data.get('feasible') is None:
                n_failed += 1          # API/parse failure — retried on next run
                continue
            if not data['feasible']:
                n_infeasible += 1      # contradictory spec — record and drop
                out_f.write(json.dumps({'syn_id': spec['syn_id'], 'feasible': False}) + '\n')
                continue
            n_ok += 1
            out_f.write(json.dumps({
                'syn_id':       spec['syn_id'],
                'nct_id':       spec['nct_id'],
                'style':        spec['style'],
                'feasible':     True,
                'patient_note': data['patient_note'],
                'targets':      spec['targets'],
            }) + '\n')

print(f'\nok: {n_ok:,}  |  infeasible: {n_infeasible:,}  |  failed (will retry): {n_failed:,}')

## Blind verification

A different model (Haiku) labels each (note, criterion) pair *without seeing the target*.
Agreement → `verified: true`. GRPO in Stage 2 should train on verified pairs only —
noisy reward labels poison the policy. The per-class agreement matrix below is also
the fidelity report for the paper.

In [ ]:
SYSTEM_VERIFY = """You are a clinical trial eligibility assessor.
Given a patient description and a single eligibility criterion, determine whether the criterion applies.

Respond with exactly one label and nothing else:
- included          (patient meets this inclusion criterion)
- not_included      (patient does not meet this inclusion criterion)
- excluded          (this exclusion criterion applies to the patient)
- not_excluded      (this exclusion criterion does not apply to the patient)
- not_enough_information  (cannot determine from the patient description)"""

VALID = {'included', 'not_included', 'excluded', 'not_excluded', 'not_enough_information'}

def parse_simple(text):
    cleaned = text.strip().lower().replace(' ', '_').strip('.')
    if cleaned in VALID:
        return cleaned
    for label in sorted(VALID, key=len, reverse=True):
        if label in cleaned:
            return label
    return 'not_enough_information'

# Build the flat pair list from feasible patients
patients = {}
with open(PATIENTS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if rec.get('feasible'):
            patients[rec['syn_id']] = rec

all_pairs = [
    {'pair_id': f"{sid}__{j}", 'syn_id': sid, 'nct_id': p['nct_id'], 'style': p['style'],
     'crit_type': t['crit_type'], 'criterion': t['criterion'], 'label': t['label']}
    for sid, p in patients.items()
    for j, t in enumerate(p['targets'])
]
print(f'Feasible patients: {len(patients):,}  |  pairs to verify: {len(all_pairs):,}')

done_pairs = set()
if os.path.exists(PAIRS_PATH):
    with open(PAIRS_PATH) as f:
        for line in f:
            done_pairs.add(json.loads(line)['pair_id'])
todo_pairs = [p for p in all_pairs if p['pair_id'] not in done_pairs]
print(f'Already verified: {len(done_pairs):,}  |  Remaining: {len(todo_pairs):,}')

vsem = asyncio.Semaphore(24)

async def verify_pair(pair):
    note = patients[pair['syn_id']]['patient_note']
    user_msg = (f"Patient: {note}\n\n"
                f"{pair['crit_type'].capitalize()} criterion: {pair['criterion']}\n\n"
                'Assess this criterion.')
    async with vsem:
        for attempt in range(3):
            try:
                resp = await aclient.messages.create(
                    model=VERIFY_MODEL, max_tokens=20, system=SYSTEM_VERIFY,
                    messages=[{'role': 'user', 'content': user_msg}],
                )
                return pair, parse_simple(resp.content[0].text)
            except Exception:
                if attempt == 2:
                    return pair, None
                await asyncio.sleep(2 ** attempt * 2)

VCHUNK = 200
for start in tqdm(range(0, len(todo_pairs), VCHUNK), desc='Verifying'):
    chunk   = todo_pairs[start:start+VCHUNK]
    results = await asyncio.gather(*[verify_pair(p) for p in chunk])
    with open(PAIRS_PATH, 'a') as out_f:
        for pair, blind in results:
            if blind is None:
                continue  # retried on next run
            out_f.write(json.dumps({**pair,
                                    'blind_label': blind,
                                    'verified': blind == pair['label']}) + '\n')

In [ ]:
pairs = []
with open(PAIRS_PATH) as f:
    for line in f:
        pairs.append(json.loads(line))

total      = len(pairs)
n_verified = sum(1 for p in pairs if p['verified'])
print(f'Total pairs    : {total:,}')
print(f'Verified (gold): {n_verified:,}  ({100*n_verified/max(total,1):.1f}%)')

print('\nAgreement rate by target class:')
by_class = Counter(p['label'] for p in pairs)
ok_class = Counter(p['label'] for p in pairs if p['verified'])
for label in sorted(by_class, key=lambda l: -by_class[l]):
    print(f'  {label:30s} {ok_class[label]:6,}/{by_class[label]:6,}  '
          f'({100*ok_class[label]/by_class[label]:.1f}%)')

print('\nConfusion (target → blind) for disagreements, top 10:')
conf = Counter((p['label'], p['blind_label']) for p in pairs if not p['verified'])
for (tgt, blind), c in conf.most_common(10):
    print(f'  {tgt:26s} → {blind:26s} {c:5,}')

print('\nVerified label distribution (GRPO training pool):')
for label, count in sorted(ok_class.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:6,}  ({100*count/max(n_verified,1):.1f}%)')
print(f'\nSaved → {PAIRS_PATH}  (join to {PATIENTS_PATH} on syn_id)')